### MULTI AGENTIC AI SYSTEM

In [ ]:
# pip install langgraph
# pip install  langgraph-supervisor
# !pip install langchain-community

In [82]:
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv 
import os 
load_dotenv()


True

### First Setup LLM

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
response = model.invoke("tell me just name, who is  prime minister of india?")
print(response.content)

Narendra Modi.


In [83]:
### SETUP 2 BASIC TOOLS 
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

In [84]:
# SETUP  WEB SEARCH API DUCK DUCK GO  

# !pip  install ddgs  
from ddgs import DDGS 
def research_tool(query:str)->str:
    """Tool to research on internet, it return information from the internet, extract the web content"""
    try:
        with DDGS() as dd:
            responses = dd.text(query)
            complete_info = ""
            for response  in responses:
                complete_info += "Title\n"
                complete_info += response['title']
                complete_info += "\n"
                complete_info += response['body']
                # complete_info += "\n"

        return complete_info 
    except Exception as e:
        return f"No result found this query {query}"

print(research_tool("Tajmahal"))


Title
Taj Mahal
The Taj Mahal ( TAHJ mə-HAHL, TAHZH -; Hindustani: [t̪ɑːd͡ʒ ˈmɛɦ(ɛ)l]; lit. 'Crown of the Palace') is an ivory-white marble mausoleum on the right bank of the river Yamuna in Agra, Uttar Pradesh, India. It was commissioned in 1631 by the fifth Mughal emperor, Shah Jahan (r. 1628–1658), to house the tomb of his beloved wife, Mumtaz Mahal; it also houses the tomb of Shah Jahan himself. The tomb is the centrepiece of a 17-hectare (42-acre) complex, which includes a mosque and a guest house, and is set in formal gardens bounded on three sides by a crenellated wall.Construction of the mausoleum was completed in 1648, but work continued on other phases of the project for another five years. The first ceremony held at the mausoleum was an observance by Shah Jahan, on 6 February 1643, of the 12th anniversary of the death of Mumtaz Mahal. The Taj Mahal complex is believed to have been completed in its entirety in 1653 at a cost estimated at the time to be around ₹32 million, whi

In [ ]:
# SETUP MATH AGENT 
math_agent = create_react_agent( 
    model=model,
    tools=[add, multiply],
    name="math_expert",
    prompt="You are a math expert. Always use one tool at a time."
)

# SETUP RESEARCHER AGENT 
research_agent = create_react_agent(
    model=model,
    tools=[research_tool],
    name="research_expert",
    prompt="You are a world class researcher with access to web search. Do not do any math. provide answer based tool response"
)

C:\Users\Ranjit\AppData\Local\Temp\ipykernel_6196\594185075.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(
C:\Users\Ranjit\AppData\Local\Temp\ipykernel_6196\594185075.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


In [86]:
# CREATE SUPERVISION AGENT  
workflow = create_supervisor(
    agents= [research_agent, math_agent],
    model=model,
    prompt=(
        "You are a team supervisor managing a research expert and a math expert. "
        "For current events, use research_agent. "
        "For math problems, use math_agent."
    )
)

# Compile and run
app = workflow.compile()


c:\Users\Ranjit\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph_supervisor\supervisor.py:431: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor_agent = create_react_agent(  # type: ignore[deprecated]


In [87]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what's the combined headcount of the FAANG companies in 2024?"
        }
    ]
})

In [80]:
result

{'messages': [HumanMessage(content="what's the combined headcount of the FAANG companies in 2024?", additional_kwargs={}, response_metadata={}, id='eebef2c8-1d59-4a8c-b8fe-4992f707cea9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 112, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DLBpxJ4ufGnr15IVi93VsjURI2Lqs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019d073a-6246-7fd3-96cb-68ab9676237b-0', tool_calls=[{'name': 'transfer_to_research_expert', 'args': {}, 'id': 'call_E0OnJCKaN6Fv35sdkIHOWti4', 'type': 'tool_call'}], invalid_tool

In [88]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

what's the combined headcount of the FAANG companies in 2024?
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_expert (call_6WwloMyE3DBTZCRbUYHKxIIQ)
 Call ID: call_6WwloMyE3DBTZCRbUYHKxIIQ
  Args:
================================= Tool Message =================================
Name: transfer_to_research_expert

Successfully transferred to research_expert
================================== Ai Message ==================================
Name: research_expert

As of 2024, the headcount for the FAANG companies (Facebook, Apple, Amazon, Netflix, and Google) is as follows:

- Facebook (Meta): Approximately 164,000
- Apple: Approximately 525,000
- Amazon: Approximately 1,183,000
- Netflix: Approximately 16,000
- Google (Alphabet): Approximately 309,000 

The combined headcount for all these companies adds up to approximately 2,19

In [91]:
# ANOTHER QUERY
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what is the weather in delhi today. Multiply it by 2 and add 5"
        }
    ]
})

In [92]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

what is the weather in delhi today. Multiply it by 2 and add 5
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_expert (call_NBucyzYYIvSx4PaWM27PvE7M)
 Call ID: call_NBucyzYYIvSx4PaWM27PvE7M
  Args:
================================= Tool Message =================================
Name: transfer_to_research_expert

Successfully transferred to research_expert
================================== Ai Message ==================================
Name: research_expert

Today, Delhi is experiencing light rainfall with a temperature of around 37°C. The weather conditions indicate some cloud cover but remain bright overall.
================================== Ai Message ==================================
Name: research_expert

Transferring back to supervisor
Tool Calls:
  transfer_back_to_supervisor (7d7757af-0336-4c0e-8f0f-db4c0104a7b3)